In [1]:
!pip install pandas sentence-transformers faiss-cpu openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os

In [3]:
file_path = "/content/Qur'an Dataset.xlsx"
df = pd.read_excel(file_path)
df.dropna(subset=["Verse"], inplace=True)

In [4]:
embeddings_file = "verse_embeddings.npy"
model_name = 'all-MiniLM-L6-v2'
if os.path.exists(embeddings_file):
    print("[INFO] Loading cached embeddings from file...")
    embeddings = np.load(embeddings_file)
    model = SentenceTransformer(model_name)
else:
    print("[INFO] Generating embeddings for all verses...")
    model = SentenceTransformer(model_name)
    embeddings = model.encode(df["Verse"].tolist(), show_progress_bar=True)
    np.save(embeddings_file, embeddings)
    print("[INFO] Embeddings saved for future use.")


[INFO] Generating embeddings for all verses...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/195 [00:00<?, ?it/s]

[INFO] Embeddings saved for future use.


In [5]:
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.array(embeddings).astype('float32'))

print("[READY] Semantic Search Initialized!")

[READY] Semantic Search Initialized!


In [ ]:
while True:
    query = input("\nEnter your search query (or type 'exit'): ")
    if query.lower() == 'exit':
        break
    query_embedding = model.encode([query])
    D, I = index.search(np.array(query_embedding).astype('float32'), 5)

    print("\nTop relevant verses:\n")
    for i in I[0]:
        verse_info = df.iloc[i]
        print(f"[Surah {verse_info['Name']} ({verse_info['Surah']}), Ayah {verse_info['Ayat']}]:\n{verse_info['Verse']}\n")



Enter your search query (or type 'exit'): zakat

Top relevant verses:

[Surah The Believers (23), Ayah 4]:
And who (always) pay Zakat (the Alms-due [and keep purifying their wealth and souls]),

[Surah Repentance (9), Ayah 60]:
Indeed, alms (Zakat) are meant for the poor and the indigent, and those who are deployed to collect charities and those in whose hearts the inculcation of love for Islam is aimed at. And, (moreover, spending Zakat for the) freeing of human lives (from the yoke of slavery) and removing the burden of those who are to pay debt and (those who toil hard) in the cause of Allah and the wayfarers (is true). This (all) has been prescribed by Allah, and Allah is All-Knowing, Most Wise.

[Surah Light (24), Ayah 56]:
And establish (the system of) prayers and (ensure) the payment of Zakat (the Alms-due) and accomplish (absolute) obedience to the Messenger (blessings and peace be upon him) so that you may be granted mercy (i.e., the blessings of sovereign rule, stability, pe